# DX 704 Week 1 Project

This week's project will build a portfolio risk and return model, and make investing recommendations for hypothetical clients.
You will collect historical data, estimate returns and risks, construct efficient frontier portfolios, and sanity check the certainty of the maximum return portfolio.

The full project description and a template notebook are available on GitHub at the following link.

https://github.com/bu-cds-dx704/dx704-project-01


Feel free to use optimization tools or libraries (such as CVXOPT or scipy.optimize) to perform any calculations required for this mini project.

### Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

In [26]:
import matplotlib.pyplot as plt
import yfinance as yf
import pandas as pd
import numpy as np
import cvxpy as cp 


## Part 1: Collect Data

Collect historical monthly price data for the last 24 months covering 6 different stocks.
The data should cover 24 consecutive months including the last month that ended before this week's material was released on Blackboard.
To be clear, if a month ends between the Blackboard release and submitting your project, you do not need to add that month.

The six different stocks must include AAPL, SPY and TSLA.
At least one of the remaining 3 tickers must start with the same letter as your last name (e.g. professor Considine could use COIN).
This is to encourage diversity in what stocks you analyze; if you discuss this project with classmates, please make sure that you pick different tickers to differentiate your work.
Do not pick stocks with fewer than 24 consecutive months of price data.

In [27]:
data = yf.download({"AAPL", "SPY", "TSLA", "WMT", "MSFT", "DIS"}, 
                   start="2024-09-01", 
                   end="2026-09-01", 
                   progress=False, 
                   auto_adjust=False)

data.head()

Price        Adj Close                                                 \
Ticker            AAPL        DIS        MSFT         SPY        TSLA   
Date                                                                    
2024-09-03  220.921936  87.098793  403.072632  539.312500  210.600006   
2024-09-04  219.017883  87.216057  402.541016  538.208801  219.410004   
2024-09-05  220.535172  86.629730  402.038971  536.899658  230.169998   
2024-09-06  218.988129  85.935913  395.453033  527.863586  210.729996   
2024-09-09  219.077377  86.326790  399.410553  533.773621  216.270004   

Price                       Close                                     ...  \
Ticker            WMT        AAPL        DIS        MSFT         SPY  ...   
Date                                                                  ...   
2024-09-03  75.786728  222.770004  89.129997  409.440002  552.080017  ...   
2024-09-04  75.855476  220.850006  89.250000  408.899994  550.950012  ...   
2024-09-05  75.580498  222.380005  88.650002  408.390015  549.609985  ...   
2024-09-06  75.266228  220.820007  87.940002  401.700012  540.359985  ...   
2024-09-09  75.953674  220.910004  88.339996  405.720001  546.409973  ...   

Price             Open                                       Volume           \
Ticker            MSFT         SPY        TSLA        WMT      AAPL      DIS   
Date                                                                           
2024-09-03  417.910004  560.469971  215.259995  77.330002  50190600  8465500   
2024-09-04  405.910004  550.200012  210.589996  77.330002  43840200  6317200   
2024-09-05  407.619995  550.890015  223.490005  77.250000  36615400  6234100   
2024-09-06  409.059998  549.940002  232.600006  76.900002  48423000  7791500   
2024-09-09  407.239990  544.650024  216.199997  76.849998  67180000  8750200   

Price                                                
Ticker          MSFT       SPY       TSLA       WMT  
Date                                                 
2024-09-03  20313600  60600100   76714200  22666100  
2024-09-04  15135800  47224900   80651800  18442800  
2024-09-05  14195500  44264300  119355000  13082900  
2024-09-06  19609500  68493800  112177000  14548600  
2024-09-09  15295100  40445800   67443500  22260200  

[5 rows x 36 columns]

Save the data as a TSV file named "historical_prices.tsv" and include a header row with the column names "date" and the 6 stock ticker symbols.
The date should be the last trading day of the month, so it may not be the last day of the month.
For example, the last trading day of November 2024 was 2024-11-29.
The remaining columns should contain the adjusted closing prices of the corresponding stock tickers on that day.


In [28]:
# Get adjusted closing prices
prices = data["Adj Close"]

# Keep the last trading day of each month
monthly_prices = prices.groupby(prices.index.to_period("M")).tail(1)

# Move the date from the index into a column
monthly_prices = monthly_prices.reset_index()
monthly_prices = monthly_prices.rename(columns={"Date": "date"})

# Put columns in the requested order
monthly_prices = monthly_prices[
    ["date", "AAPL", "SPY", "TSLA", "WMT", "MSFT", "DIS"]
]

# Save as TSV
monthly_prices.to_csv(
    "historical_prices.tsv",
    sep="\t",
    index=False
)

monthly_prices

Ticker,date,AAPL,SPY,TSLA,WMT,MSFT,DIS
0,2024-09-30,231.067062,562.210449,261.630005,79.302567,423.608276,93.997910
1,2024-10-31,224.035873,557.193604,249.850006,80.481049,400.030731,94.007668
2,2024-11-29,235.620117,590.420898,345.160004,90.841949,417.709076,114.792938
3,2024-12-31,248.615799,576.215271,403.839996,88.927063,415.775696,109.294563
4,2025-01-31,234.299667,591.690308,404.600006,96.614075,409.423157,110.973007
5,2025-02-28,240.361572,584.178894,292.980011,97.056976,392.383728,111.699348
6,2025-03-31,220.772095,551.628906,259.160004,86.644676,371.034393,96.878075
7,2025-04-30,211.200943,546.846191,282.160004,95.981247,390.673859,89.271141
8,2025-05-30,199.883957,581.212769,346.459991,97.667648,455.853821,110.953377
9,2025-06-30,204.183151,611.079102,317.660004,96.737663,492.541168,122.239944


Submit "historical_prices.tsv" in Gradescope.

## Part 2: Calculate Historical Asset Returns

Calculate the historical asset returns based on the price data that you previously collected.

In [29]:
historical_prices = pd.read_csv("historical_prices.tsv", sep="\t")
historical_prices = historical_prices.set_index("date")

historical_returns = historical_prices.pct_change().dropna()
historical_returns

,AAPL,SPY,TSLA,WMT,MSFT,DIS
date,,,,,,
2024-10-31,-0.030429,-0.008923,-0.045025,0.014861,-0.055659,0.000104
2024-11-29,0.051707,0.059633,0.381469,0.128737,0.044192,0.221102
2024-12-31,0.055155,-0.024060,0.170008,-0.021079,-0.004629,-0.047898
2025-01-31,-0.057583,0.026856,0.001882,0.086442,-0.015279,0.015357
2025-02-28,0.025872,-0.012695,-0.275877,0.004584,-0.041618,0.006545
2025-03-31,-0.081500,-0.055719,-0.115435,-0.107280,-0.054409,-0.132689
2025-04-30,-0.043353,-0.008670,0.088748,0.107757,0.052932,-0.078521
2025-05-30,-0.053584,0.062845,0.227885,0.017570,0.166840,0.242881
2025-06-30,0.021508,0.051386,-0.083126,-0.009522,0.080481,0.101724


Save the data as a TSV file named "historical_returns.tsv" and include a header row with the column names "date" and the 6 stock ticker symbols.
Each row should have the date at the end of the month and the corresponding *relative* price changes.
For example, if the previous price was \$100 and the new price is \$110, the return value should be 0.10.
There should only be 23 rows of data in this file, since they are computed as the differences of 24 prices.

In [30]:
# Move the date from the index into a column
historical_returns_tsv = historical_returns.reset_index()
historical_returns_tsv = historical_returns_tsv.rename(columns={"Date": "date"})

# Put columns in the requested order
historical_returns_tsv = historical_returns_tsv[
    ["date", "AAPL", "SPY", "TSLA", "WMT", "MSFT", "DIS"]
]

# Save as TSV
historical_returns_tsv.to_csv(
    "historical_returns.tsv",
    sep="\t",
    index=False
)

Submit "historical_returns.tsv" in Gradescope.

## Part 3: Estimate Returns

Estimate the expected returns for each asset using the previously calculated return data.
Just compute the average (mean) return for each asset over your data set; do not use other estimators that have been mentioned.
This will serve as your estimate of expected return for each asset.

In [31]:
estimated_returns = historical_returns.mean()
estimated_returns

AAPL    0.015696
SPY     0.014246
TSLA    0.027043
WMT     0.014172
MSFT    0.012032
DIS     0.009831
dtype: float64

Save the estimated returns in a TSV file named "estimated_returns.tsv" and include a header row with the column names "asset" and "estimated_return".

In [32]:
# Save as TSV
estimated_returns = estimated_returns.reset_index()
estimated_returns.columns = ["asset", "estimated_return"]

estimated_returns.to_csv(
    "estimated_returns.tsv",
    sep="\t",
    index=False
)

Submit "estimated_returns.tsv" in Gradescope.

## Part 4: Estimate Risk

Estimate the covariance matrix for the asset returns to understand how the assets move together.

In [33]:
estimated_correlations = historical_returns.corr()

estimated_covariance = historical_returns.cov()
estimated_covariance

,AAPL,SPY,TSLA,WMT,MSFT,DIS
AAPL,0.004014,0.001053,0.002829,-0.000504,0.002296,0.000439
SPY,0.001053,0.001380,0.002818,0.000625,0.001843,0.002407
TSLA,0.002829,0.002818,0.026124,0.002304,0.002710,0.007064
WMT,-0.000504,0.000625,0.002304,0.004080,-0.000654,0.001280
MSFT,0.002296,0.001843,0.002710,-0.000654,0.008978,0.004237
DIS,0.000439,0.002407,0.007064,0.001280,0.004237,0.008826


Save the estimated covariances to a TSV file named "estimated_covariance.tsv".
The header row should have a blank column name followed by the names of the assets.
Each data row should start with the name of an asset for that row, and be followed by the individual covariances corresponding to that row and column's assets.
(This is the format of pandas's `to_csv` method with `sep="\t"` when used on a covariance matrix as computed in the examples.)

In [34]:
estimated_covariance.to_csv(
    "estimated_covariance.tsv",
    sep="\t"
)

Submit "estimated_covariance.tsv" in Gradescope.

## Part 5: Construct the Maximum Return Portfolio

Compute the maximum return portfolio based on your previously estimated risks and returns.

In [35]:
n = len(estimated_returns)
x = cp.Variable(n)

objective = cp.Maximize(
    estimated_returns["estimated_return"].to_numpy() @ x
)

prob = cp.Problem(
    objective,
    [
        0 <= x,
        cp.sum(x) == 1
    ]
)

estimated_return_P = prob.solve()

x.value.round(2)

portfolio = pd.DataFrame({
    "asset": estimated_returns["asset"],
    "allocation": x.value.round(2)
})

portfolio

,asset,allocation
0,AAPL,0.0
1,SPY,0.0
2,TSLA,1.0
3,WMT,0.0
4,MSFT,0.0
5,DIS,0.0


Save the maximum return portfolio in a TSV file named "maximum_return.tsv".
The header row should have two columns, "asset" and "allocation".
The allocation values should sum up to one.


In [36]:
portfolio.to_csv(
    "maximum_return.tsv",
    sep="\t"
)

Submit "maximum_return.tsv" in Gradescope.

## Part 6: Construct the Minimum Risk Portfolio

Compute the minimum risk portfolio based on your previously estimated risks.

In [37]:
x_minimum_risk = cp.Variable(n) 

objective_min_risk = cp.Minimize(x_minimum_risk.T @ estimated_covariance.to_numpy() @x_minimum_risk) 

prob_min_risk = cp.Problem(objective_min_risk, 
                           [0 <= x_minimum_risk, 
                            cp.sum(x_minimum_risk) == 1]) 

covariance_min_risk = prob_min_risk.solve()

minimum_risk_portfolio = pd.DataFrame({
    "asset": estimated_returns["asset"],
    "allocation": x_minimum_risk.value
})

minimum_risk_portfolio

,asset,allocation
0,AAPL,1.500713e-01
1,SPY,6.420884e-01
2,TSLA,-5.778991e-20
3,WMT,2.078402e-01
4,MSFT,-1.110491e-19
5,DIS,-1.777567e-19


Save the minimum risk portfolio in a TSV file named "minimum_risk.tsv".
The header row should have two columns, "asset" and "allocation".
The allocation values should sum up to one.


In [38]:
minimum_risk_portfolio.to_csv(
    "minimum_risk.tsv",
    sep="\t"
)

Submit "minimum_risk.tsv" in Gradescope.

## Part 7: Build Efficient Frontier Portfolios

Compute 101 portfolios along the mean-variance efficient frontier with evenly spaced estimated returns.
The first portfolio should be the minimum risk portfolio from part 4, and the last portfolio should be the maximum return portfolio from part 3.
The estimated return of each portfolio should be higher than the previous by one percent of the difference between the first and last portfolios.
That is, the estimated return of the portfolios should be similar to `np.linspace(min_risk_return, max_return, 101)`.


In [39]:
estimated_returns = historical_returns.mean()

estimated_return_min_risk = x_minimum_risk.value @ estimated_returns
estimated_return_min_risk

np.float64(0.014448152467384645)

In [40]:
estimated_return_max_return = estimated_returns.max()
estimated_return_max_return

np.float64(0.027043245645209116)

In [41]:
ef_variances = []
ef_returns = []
ef_portfolios = []

for r in np.linspace(estimated_return_min_risk, estimated_return_max_return, 101):
    x_r = cp.Variable(n)

    prob_r = cp.Problem(cp.Minimize(x_r.T @ estimated_covariance.to_numpy() @ x_r),
                        [0 <= x_r, cp.sum(x_r) == 1, estimated_returns.to_numpy().reshape(1, -1) @ x_r == r])

    ef_variances.append(prob_r.solve())
    ef_returns.append(r)
    ef_portfolios.append(x_r.value)

ef_portfolios = np.asarray(ef_portfolios)

efficient_frontier = pd.DataFrame({
    "index": range(101),
    "return": ef_returns,
    "risk": np.sqrt(ef_variances),
    **{
        asset: ef_portfolios[:, i]
        for i, asset in enumerate(estimated_returns.index)
    }
})

efficient_frontier

,index,return,risk,AAPL,SPY,TSLA,WMT,MSFT,DIS
0,0,0.014448,0.034261,1.500713e-01,6.420884e-01,1.613712e-19,2.078402e-01,-5.678468e-20,-7.766760e-20
1,1,0.014574,0.034569,2.081073e-01,5.721202e-01,3.316516e-03,2.164560e-01,-8.153901e-20,-1.387836e-19
2,2,0.014700,0.035059,2.142272e-01,5.564854e-01,1.246730e-02,2.168202e-01,-7.715872e-20,-1.312806e-19
3,3,0.014826,0.035598,2.203470e-01,5.408506e-01,2.161808e-02,2.171844e-01,-7.278482e-20,-1.237849e-19
4,4,0.014952,0.036185,2.264669e-01,5.252157e-01,3.076886e-02,2.175485e-01,-6.841157e-20,-1.162941e-19
...,...,...,...,...,...,...,...,...,...
96,96,0.026539,0.155254,4.439887e-02,-8.722412e-24,9.556011e-01,-9.786431e-24,-4.024263e-24,-2.407641e-23
97,97,0.026665,0.156844,3.329915e-02,-2.237284e-23,9.667008e-01,-2.146074e-23,-3.132168e-23,-4.250989e-23
98,98,0.026791,0.158436,2.219944e-02,-8.525722e-25,9.778006e-01,-1.720916e-24,-1.557718e-23,5.145152e-26
99,99,0.026917,0.160032,1.109972e-02,-1.366010e-23,9.889003e-01,-1.005479e-23,-3.204392e-23,-4.574387e-23


Save the portfolios in a TSV file named "efficient_frontier.tsv".
The header row should have columns "index", "return", "risk", and all the asset tickers.
Each data row should have the portfolio index (0-100), the estimated return of the portfolio, the estimated standard deviation (not variance) of the portfolio, and all the asset allocations (which should sum to one).

In [42]:
efficient_frontier.to_csv(
    "efficient_frontier.tsv",
    sep="\t"
)

Submit "efficient_frontier.tsv" in Gradescope.

## Part 8: Check Maximum Return Portfolio Stability

Check the stability of the maximum return portfolio by resampling the estimated risk/return model.

Repeat 1000 times -
1. Use `np.random.multivariate_normal` to generate 23 return samples using your previously estimated risks and returns.
2. Estimate the return of each asset using that resampled return history.
3. Check which asset had the highest return in those resampled estimates.

This procedure is a reduced and simplified version of the Michaud resampled efficient frontier procedure that takes uncertainty in the risk model into account.

In [43]:
max_return_assets = []

mean_returns = estimated_returns.to_numpy()
cov_matrix = estimated_covariance.to_numpy()

for i in range(1000):
    
    # Generate 23 simulated monthly return observations
    resampled_returns = np.random.multivariate_normal(
        mean_returns,
        cov_matrix,
        size=23
    )
    
    # Estimate each asset's return from the simulated history
    resampled_estimated_returns = resampled_returns.mean(axis=0)
    
    # Find which asset has the highest estimated return
    max_asset_index = np.argmax(resampled_estimated_returns)
    
    # Save the ticker
    max_return_assets.append(
        estimated_returns.index[max_asset_index]
    )

In [44]:
max_return_probabilities = (
    pd.Series(max_return_assets)
    .value_counts(normalize=True)
    .reindex(estimated_returns.index, fill_value=0)
    .reset_index()
)

max_return_probabilities.columns = ["asset", "probability"]

max_return_probabilities

,asset,probability
0,AAPL,0.128
1,SPY,0.025
2,TSLA,0.487
3,WMT,0.139
4,MSFT,0.139
5,DIS,0.082


Save a file "max_return_probabilities.tsv" with the distribution of highest return assets.
The header row should have columns "asset" and "probability".
There should be a data row for each asset and its sample probability of having the highest return based on those 1000 resampled estimates.


In [45]:
max_return_probabilities.to_csv(
    "max_return_probabilities.tsv",
    sep="\t"
)

Submit "max_return_probabilities.tsv" in Gradescope.

## Part 9: Acknowledgments

Make a file "acknowledgments.txt" documenting any outside sources or help on this project.
If you discussed this assignment with anyone, please acknowledge them here.
If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for.
If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy.
If no acknowledgments are appropriate, just write none in the file.


Submit "acknowledgments.txt" in Gradescope.

In [46]:
acknowledgments = """Acknowledgments

I used ChatGPT (OpenAI) for assistance with debugging Python code, interpreting error messages, and help with code for tsv file sharing.
ChatGPT transcript: https://chatgpt.com/share/6a9df652-4a48-83e9-8621-c0393d4ecc4a
"""

with open("acknowledgments.txt", "w") as file:
    file.write(acknowledgments)

## Part 10: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

Submit "project.ipynb" in Gradescope.